In [1]:
import pandas as pd
from datetime import datetime
import sqlalchemy
from sqlalchemy import create_engine

In [2]:
env = {
    'paths': ['raw/03_Library Systembook.csv','raw/03_Library SystemCustomers.csv','logs'],
    'element': ['library_stuff', 'library_users','library_logs'],
    'rows_dropped' : [None,None,None],
    'rows_input' : [None,None,None],
    'rows_output' : [None,None,None],
    'start_time': [None,None,None],
    'end_time': [None,None,None],
}

In [3]:
ingest_loans_dtypes = {
    'Id':"Int64",
    'Books': str,
    'Days allowed to borrow':str,
    'Customer ID':"Int64"
}

In [4]:
def storage(frame, location):
    frame.to_csv(location,index=False)

def drop_null_records(df):
    return df.dropna()

def rename_cols(frame):
    old_cols = frame.columns
    for c in frame.columns:
        frame[c.replace(' ','').lower()] = frame[c]

    frame = frame.drop(old_cols, axis=1)
    return frame

def clean_dates(frame, date_cols=None):
    if date_cols == None:
        return frame
    df = frame.map(lambda x: x.replace('"', '') if isinstance(x, str) else x)
    df = df.map(lambda x: x.replace('"', '') if isinstance(x, str) else x)
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df 

def store_in_db(df, tablename):
    server = "localhost"
    database = "library_stuff"

    connection_string = (
            f"mssql+pyodbc://@{server}/{database}"
                "?driver=ODBC+Driver+17+for+SQL+Server"
                    "&trusted_connection=yes"
                    )

    engine = create_engine(connection_string)
    df.to_sql(tablename, engine, if_exists="append",   index=False)

In [ ]:
def overlord(env_var=env, element_index=0):
    env_var['start_time'][element_index] = datetime.now()
    if element_index == 2:
        print(element_index)
        env_var['rows_input'][element_index] = 1
        env_var['rows_output'][element_index] = 1
        
        env_var['rows_dropped'][element_index] = 0
        logs = pd.DataFrame(env_var)
        logs = logs[logs['paths']!='logs']
        store_in_db(logs, env_var['element'][element_index])
        return logs
    
    if element_index == 0:
        date_cols = ['Book checkout','Book Returned']
    else:
        date_cols = None

    df = pd.read_csv(env_var['paths'][element_index])
    df['rundate'] = datetime.now().strftime('%Y%m%d')
    print(df.iloc[0])
    env_var['rows_input'][element_index] = len(df)
    df= clean_dates(df,date_cols)
    df=  rename_cols(df)
    df =  drop_null_records(df)
    env_var['rows_dropped'][element_index] = env_var['rows_input'][element_index]-len(df)
    env_var['end_time'][element_index] = datetime.now()
    env_var['rows_output'][element_index] = len(df)
    store_in_db(df, env_var['element'][element_index])
    return df

for i in range(3):
    # 1and 2 run ingest and clean, 3 stores logs
    test_load = overlord(env, i)


Id                                        1.0
Books                     Catcher in the Rye 
Book checkout                    "20/02/2023"
Book Returned                      25/02/2023
Days allowed to borrow                2 weeks
Customer ID                               1.0
rundate                            2026-06-01
Name: 0, dtype: object


C:\Users\Admin\AppData\Local\Temp\ipykernel_19796\2876305040.py:21: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Admin\AppData\Local\Temp\ipykernel_19796\2876305040.py:21: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')


Customer ID             1.0
Customer Name      Jane Doe
rundate          2026-06-01
Name: 0, dtype: object
2
